<a href="https://colab.research.google.com/github/xmayksx1/DAX-BI/blob/main/Consumo_de_Inventario_ejemplo_PMI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta
import os

# --- 1. Carga de Datos ---
carga = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Documentos\PMI.xlsx"
df = pd.read_excel(carga, sheet_name='Demanda')


df.columns = df.columns.str.strip()

df2 = pd.read_excel(carga, sheet_name='Inventario')

df2['Fecha Vence'] = pd.to_datetime(df2['Fecha Vence'])
df2 = df2.sort_values(by=['Código LN', 'Fecha Vence']).reset_index(drop=True)
inv_operativo = df2.copy()

# Diccionario para mostrar los meses en español en los reportes
meses_es = {1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun',
            7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'}

# Extraer el mes y año de vencimiento real para análisis (Ej: 2026-07)
inv_operativo['Mes_Vencimiento'] = inv_operativo['Fecha Vence'].dt.strftime('%Y-%m')

# --- 2. Parámetros de Simulación ---
num_semanas = 18
UMBRAL_PRONTO_VENCE = 30
VALOR_THS = 35.03
fecha_hoy = datetime.now() # Toma la fecha actual automáticamente

resultados_riesgo = []

# --- 3. Simulación ---
for semana in range(1, num_semanas + 1):
    fecha_simulada = fecha_hoy + timedelta(days=(semana * 7))

    # Calculamos el mes real proyectado basado en la fecha de simulación
    # Formato: "2026-05 (May)" para garantizar que se ordene cronológicamente
    mes_proyectado = f"{fecha_simulada.year}-{fecha_simulada.month:02d} ({meses_es[fecha_simulada.month]})"

    # Evaluar riesgo en la semana actual (Fotografía)
    for idx_inv, lote in inv_operativo[inv_operativo['Inventario WMS'] > 0].iterrows():
        dias_remanentes = (lote['Fecha Vence'] - fecha_simulada).days

        if dias_remanentes <= UMBRAL_PRONTO_VENCE:
            resultados_riesgo.append({
                'Semana': semana,
                'Fecha_Corte': fecha_simulada.strftime('%Y-%m-%d'),
                'Mes_Simulado': mes_proyectado,
                'Mes_Vencimiento_Real': lote['Mes_Vencimiento'],
                'ITMCodigo': lote['Código LN'],
                'Lote': lote['Lote'],
                'THS_Riesgo': lote['Inventario WMS'],
                'Dinero_Riesgo_USD': lote['Inventario WMS'] * VALOR_THS
            })

    # Consumir la demanda (BLOQUEANDO LOTES <= 90 DÍAS)
    for _, fila_dem in df.iterrows():
        sku = fila_dem['ITMCodigo']
        dem_sem = fila_dem['Promedio de las últimas semanas']

        # Filtramos para que la demanda solo tome lotes mayores a 90 días
        lotes_aptos = inv_operativo[
            (inv_operativo['Código LN'] == sku) &
            (inv_operativo['Inventario WMS'] > 0) &
            ((inv_operativo['Fecha Vence'] - pd.to_datetime(fecha_simulada)).dt.days > UMBRAL_PRONTO_VENCE)
        ].index

        for idx in lotes_aptos:
            if dem_sem <= 0: break
            stock = inv_operativo.at[idx, 'Inventario WMS']
            consumo = min(dem_sem, stock)
            inv_operativo.at[idx, 'Inventario WMS'] -= consumo
            dem_sem -= consumo

df_riesgo = pd.DataFrame(resultados_riesgo)

# --- 4. Agrupaciones Mensuales ---

# A. PROYECCIÓN MENSUAL: Tomamos el inventario en riesgo de la última semana que cae dentro de cada mes calendario
semanas_fin_de_mes = df_riesgo.groupby('Mes_Simulado')['Semana'].transform('max')
df_riesgo_mensual_snapshot = df_riesgo[df_riesgo['Semana'] == semanas_fin_de_mes]

# Para asegurarnos de que el orden de los meses sea correcto en la gráfica y el excel
resumen_mensual = df_riesgo_mensual_snapshot.groupby('Mes_Simulado').agg({
    'Dinero_Riesgo_USD': 'sum',
    'THS_Riesgo': 'sum',
    'Semana': 'max' # Guardamos la semana para ordenar cronológicamente
}).reset_index().sort_values('Semana')

# B. RIESGO POR MES DE VENCIMIENTO (Estado final del inventario en la semana 12)
df_final_simulacion = df_riesgo[df_riesgo['Semana'] == num_semanas]
riesgo_por_mes_vencimiento = df_final_simulacion.groupby('Mes_Vencimiento_Real').agg({
    'Dinero_Riesgo_USD': 'sum',
    'THS_Riesgo': 'sum'
}).reset_index().sort_values('Mes_Vencimiento_Real')

# --- 5. Gráfico Mensual Plotly ---
fig = go.Figure()

# Barras para Dinero
fig.add_trace(go.Bar(
    x=resumen_mensual['Mes_Simulado'],
    y=resumen_mensual['Dinero_Riesgo_USD'],
    name='Dinero en Riesgo (USD)',
    marker_color='#002D62',
    yaxis='y1',
    text=resumen_mensual['Dinero_Riesgo_USD'].apply(lambda x: f'${x:,.0f}'),
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(color="white")
))

# Línea para Volumen
fig.add_trace(go.Scatter(
    x=resumen_mensual['Mes_Simulado'],
    y=resumen_mensual['THS_Riesgo'],
    name='Volumen THS',
    mode='lines+markers+text',
    line=dict(color='#FF4B4B', width=3),
    yaxis='y2',
    text=resumen_mensual['THS_Riesgo'].apply(lambda x: f'{x:,.0f}'),
    textposition='top center',
    textfont=dict(color="#FF4B4B", size=12)
))

fig.update_layout(
    title='Proyección Mensual de Riesgo de Inventario: USD vs THS',
    template='plotly_white',
    xaxis=dict(title='Mes Calendario Proyectado', showgrid=False),
    yaxis=dict(
        title='Monto en Riesgo (USD)',
        side='left',
        showgrid=True,
        gridcolor='LightGray',
        tickformat='$,.0f'
    ),
    yaxis2=dict(
        title='Cantidad (THS)',
        side='right',
        overlaying='y',
        showgrid=False
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    font=dict(family="Arial", size=12),
    margin=dict(t=100)
)

# --- 6. Exportación a Excel ---
ruta_excel = r"C:\Users\Usuario\OneDrive - Aceitera El Real\Documentos\Reporte_Riesgo_Vencimiento_Mensual.xlsx"

try:
    fig.write_image("grafico_mensual.png", scale=2, width=1200, height=600)
    grafico_disponible = True
except Exception as e:
    print(f"No se pudo generar la imagen del gráfico: {e}")
    grafico_disponible = False

with pd.ExcelWriter(ruta_excel, engine='xlsxwriter') as writer:
    # 1. Resumen por mes proyectado calendario (Ej: 2026-05, 2026-06)
    resumen_mensual[['Mes_Simulado', 'THS_Riesgo', 'Dinero_Riesgo_USD']].to_excel(writer, sheet_name='Proyeccion_Calendario', index=False)

    # 2. Resumen por mes de vencimiento (Cuándo caduca la mercancía restante)
    riesgo_por_mes_vencimiento.to_excel(writer, sheet_name='Riesgo_Mes_Vencimiento', index=False)

    # 3. Detalle final de los lotes que quedaron en riesgo en la última semana
    detalle_pivot = df_final_simulacion.pivot_table(
        index=['ITMCodigo', 'Lote', 'Mes_Vencimiento_Real'], values='Dinero_Riesgo_USD', aggfunc='sum'
    ).fillna(0).reset_index()
    detalle_pivot.to_excel(writer, sheet_name='Detalle_Lotes_Final', index=False)

    if grafico_disponible:
        workbook = writer.book
        worksheet = workbook.add_worksheet('Grafico_Financiero')
        worksheet.insert_image('B2', 'grafico_mensual.png')

print(f"Reporte mensual generado con éxito en: {ruta_excel}")
fig.show()